In [ ]:
import os, sys, json, subprocess, time, io, gc, shutil, glob, base64
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
import requests as http_requests

NOTEBOOK_NAME = "renderizador-kaggle-pt-2"
STEP_NAME = "step_render_pt2"

print("Instalando FFmpeg...")
os.system("apt-get install -y ffmpeg > /dev/null 2>&1")

print("Baixando pacote de fontes...")
os.system("mkdir -p /usr/share/fonts/truetype/custom")
# Baixando fontes populares (Montserrat, Inter, Roboto, Bebas Neue, Oswald) do Google Fonts (via github raw links or direct)
fonts = [
    ("Montserrat-Regular.ttf", "https://github.com/JulietaUla/Montserrat/raw/master/fonts/ttf/Montserrat-Regular.ttf"),
    ("Montserrat-Bold.ttf", "https://github.com/JulietaUla/Montserrat/raw/master/fonts/ttf/Montserrat-Bold.ttf"),
    ("Montserrat-Black.ttf", "https://github.com/JulietaUla/Montserrat/raw/master/fonts/ttf/Montserrat-Black.ttf"),
    ("Montserrat-Italic.ttf", "https://github.com/JulietaUla/Montserrat/raw/master/fonts/ttf/Montserrat-Italic.ttf"),
    ("Inter-Regular.ttf", "https://github.com/rsms/inter/raw/master/docs/font-files/Inter-Regular.ttf"),
    ("Inter-Bold.ttf", "https://github.com/rsms/inter/raw/master/docs/font-files/Inter-Bold.ttf"),
    ("Roboto-Regular.ttf", "https://github.com/google/fonts/raw/main/apache/roboto/static/Roboto-Regular.ttf"),
    ("Roboto-Bold.ttf", "https://github.com/google/fonts/raw/main/apache/roboto/static/Roboto-Bold.ttf"),
    ("BebasNeue-Regular.ttf", "https://github.com/dharmatype/Bebas-Neue/raw/master/fonts/BebasNeue(version1.400).ttf"),
    ("Oswald-Regular.ttf", "https://github.com/googlefonts/OswaldFont/raw/master/fonts/ttf/Oswald-Regular.ttf"),
    ("Oswald-Bold.ttf", "https://github.com/googlefonts/OswaldFont/raw/master/fonts/ttf/Oswald-Bold.ttf"),
    ("TitanOne-Regular.ttf", "https://github.com/google/fonts/raw/main/ofl/titanone/TitanOne-Regular.ttf"),
    ("LuckiestGuy-Regular.ttf", "https://github.com/google/fonts/raw/main/ofl/luckiestguy/LuckiestGuy-Regular.ttf"),
    ("FredokaOne-Regular.ttf", "https://github.com/google/fonts/raw/main/ofl/fredokaone/FredokaOne-Regular.ttf"),
    ("Bangers-Regular.ttf", "https://github.com/google/fonts/raw/main/ofl/bangers/Bangers-Regular.ttf")
]
for fname, url in fonts:
    os.system(f"wget -qO /usr/share/fonts/truetype/custom/{fname} {url}")
os.system("fc-cache -f -v > /dev/null 2>&1")
print("Fontes instaladas!")
def _load_secrets():
    try:
        from kaggle_secrets import UserSecretsClient
        _s = UserSecretsClient()
        def _get(name):
            try: return _s.get_secret(name)
            except: return ""
        print("Carregando Kaggle Secrets...")
        return _get
    except ImportError:
        from dotenv import load_dotenv; load_dotenv()
        return lambda name: os.getenv(name, "")

_get = _load_secrets()
DRIVE_REFRESH_TOKEN = _get("DRIVE_REFRESH_TOKEN")
DRIVE_CLIENT_ID = _get("DRIVE_CLIENT_ID")
DRIVE_CLIENT_SECRET = _get("DRIVE_CLIENT_SECRET")
DATABASE_URL = _get("DATABASE_URL")
PROJECT_ID = _get("PIPELINE_PROJECT_ID")
PIPELINE_WEBHOOK_URL = _get("PIPELINE_WEBHOOK_URL")

print("Autenticando Drive...")
try:
    _creds = Credentials(token=None, refresh_token=DRIVE_REFRESH_TOKEN,
        token_uri="https://oauth2.googleapis.com/token",
        client_id=DRIVE_CLIENT_ID, client_secret=DRIVE_CLIENT_SECRET,
        scopes=["https://www.googleapis.com/auth/drive"])
    _creds.refresh(Request())
    drive_service = build("drive", "v3", credentials=_creds)
    print("Drive OK!")
except Exception as e:
    drive_service = None
    print(f"Drive falhou: {e}")

def _buscar_id(caminho):
    partes = caminho.strip("/").split("/")
    pid = "root"
    for p in partes:
        q = f"name=\'{p}\' and \'{pid}\' in parents and trashed=false"
        r = drive_service.files().list(q=q, fields="files(id,mimeType)").execute()
        a = r.get("files", [])
        if not a: return None
        pid = a[0]["id"]
    return pid

def _garantir_pasta(caminho):
    partes = caminho.strip("/").split("/")
    pid = "root"
    for p in partes:
        q = f"name=\'{p}\' and \'{pid}\' in parents and trashed=false and mimeType=\'application/vnd.google-apps.folder\'"
        r = drive_service.files().list(q=q, fields="files(id)").execute()
        e = r.get("files", [])
        if e: pid = e[0]["id"]
        else:
            nova = drive_service.files().create(body={"name": p, "mimeType": "application/vnd.google-apps.folder", "parents": [pid]}, fields="id").execute()
            pid = nova["id"]
    return pid

def baixar_do_drive(caminho_drive, destino_local):
    if os.path.exists(destino_local): return True
    try:
        fid = _buscar_id(caminho_drive)
        if not fid: print(f"  Nao encontrado: {caminho_drive}"); return False
        req = drive_service.files().get_media(fileId=fid)
        os.makedirs(os.path.dirname(destino_local) or ".", exist_ok=True)
        with open(destino_local, "wb") as fh:
            dl = MediaIoBaseDownload(fh, req); done = False
            while not done: _, done = dl.next_chunk()
        print(f"  Baixado: {caminho_drive}"); return True
    except Exception as ex: print(f"  Erro: {ex}"); return False

def salvar_no_drive(caminho_local, caminho_drive):
    if not drive_service or not os.path.exists(caminho_local): return
    try:
        partes = caminho_drive.strip("/").split("/")
        nome = partes[-1]; pasta = "/".join(partes[:-1]) if len(partes) > 1 else ""
        pid = _garantir_pasta(pasta) if pasta else "root"
        q = f"name=\'{nome}\' and \'{pid}\' in parents and trashed=false"
        r = drive_service.files().list(q=q, fields="files(id)").execute()
        e = r.get("files", []); media = MediaFileUpload(caminho_local, resumable=True)
        if e: drive_service.files().update(fileId=e[0]["id"], media_body=media).execute()
        else: drive_service.files().create(body={"name": nome, "parents": [pid]}, media_body=media, fields="id").execute()
        print(f"  Salvo: {caminho_drive}")
    except Exception as ex: print(f"  Erro salvar: {ex}")

_cell_timers = {}
def _db_exec(query, params):
    if not DATABASE_URL: return
    try:
        import psycopg2; conn = psycopg2.connect(DATABASE_URL); cur = conn.cursor()
        cur.execute(query, params); conn.commit(); cur.close(); conn.close()
    except: pass

def cell_start(idx, name=""):
    _cell_timers[idx] = time.time()
    print(f"\n{'='*50}\n  CELULA [{idx}] {name}\n{'='*50}")
    if not PROJECT_ID: return
    _db_exec("INSERT INTO pipeline_cell_tracking (project_id,notebook,cell_index,cell_name,status,started_at) VALUES (%s::uuid,%s,%s,%s,\'running\',NOW()) ON CONFLICT DO NOTHING", (PROJECT_ID, NOTEBOOK_NAME, idx, name))
    _db_exec("UPDATE pipeline_cell_tracking SET status=\'running\',started_at=NOW(),finished_at=NULL,cell_name=%s WHERE project_id=%s::uuid AND notebook=%s AND cell_index=%s", (name, PROJECT_ID, NOTEBOOK_NAME, idx))

def cell_end(idx, status="done", msg=""):
    elapsed = ""
    if idx in _cell_timers:
        s = int(time.time() - _cell_timers.pop(idx))
        elapsed = f" ({s//60}m{s%60}s)" if s >= 60 else f" ({s}s)"
    icon = "OK" if status == "done" else "ERRO"
    print(f"  [{icon}] CELULA [{idx}] {status}{elapsed}: {msg}\n{'_'*50}")
    if not PROJECT_ID: return
    _db_exec("UPDATE pipeline_cell_tracking SET status=%s,finished_at=NOW(),duration_seconds=EXTRACT(EPOCH FROM(NOW()-started_at)),message=%s WHERE project_id=%s::uuid AND notebook=%s AND cell_index=%s", (status, msg, PROJECT_ID, NOTEBOOK_NAME, idx))

def report_step(status, msg=""):
    if PROJECT_ID and PIPELINE_WEBHOOK_URL:
        try:
            http_requests.post(f"{PIPELINE_WEBHOOK_URL}/webhook/status", json={"project_id": PROJECT_ID, "step": STEP_NAME, "status": status, "message": msg}, timeout=15)
        except: pass
    print(f"\nNOTEBOOK FINALIZADO: {STEP_NAME} -> {status}")
    if not PROJECT_ID: return
    _db_exec(f"UPDATE pipeline_projects SET {STEP_NAME}=%s,current_step=%s,updated_at=NOW() WHERE id=%s::uuid", (status, STEP_NAME.replace("step_",""), PROJECT_ID))

cell_start(0, "Setup e Autenticacao")
DRIVE_ATIVO = "KAGGLE/PIPELINE/ATIVO"
DRIVE_ENHANCER = "KAGGLE/PIPELINE/ENHANCER"
DRIVE_OMNI = "KAGGLE/PIPELINE/OMNI"
DRIVE_RENDER = "KAGGLE/PIPELINE/RENDER"
BASE_PATH = "/kaggle/working"
os.makedirs(BASE_PATH, exist_ok=True)

cell_end(0, "done", "Setup concluido")


In [ ]:
cell_start(1, "Download Arquivos do Drive")

# Baixar video enhanced + config + legendas + audio do Drive
baixar_do_drive(f"{DRIVE_ENHANCER}/pt2_enhanced.mp4", f"{BASE_PATH}/pt2_enhanced.mp4")
baixar_do_drive(f"{DRIVE_OMNI}/videorender-project.json", f"{BASE_PATH}/videorender-project.json")
baixar_do_drive(f"{DRIVE_OMNI}/legendas.ass", f"{BASE_PATH}/legendas.ass")
baixar_do_drive(f"{DRIVE_OMNI}/audio_dublado.mp3", f"{BASE_PATH}/audio_dublado.mp3")

# Carregar config
import shutil
baixar_do_drive(f"{DRIVE_ATIVO}/split_info.json", f"{BASE_PATH}/split_info.json")

import json
with open(f"{BASE_PATH}/split_info.json", "r") as f:
    split_info = json.load(f)

part_idx = 2
start_time = (part_idx - 1) * split_info["part_duration"]
print(f"Start time para pt{part_idx}: {start_time}")

def parse_ass_time(t_str):
    h, m, s = t_str.split(':')
    s, cs = s.split('.')
    return int(h)*3600 + int(m)*60 + int(s) + int(cs)/100.0

def format_ass_time(secs):
    if secs < 0: secs = 0
    h = int(secs // 3600)
    m = int((secs % 3600) // 60)
    s = int(secs % 60)
    cs = int(round((secs - int(secs)) * 100))
    if cs == 100:
        s += 1; cs = 0
    return f"{h}:{m:02d}:{s:02d}.{cs:02d}"

out = []
with open(f"{BASE_PATH}/legendas.ass", "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith('Dialogue:'):
            parts = line.split(',', 9)
            start_ms = parse_ass_time(parts[1]) - start_time
            end_ms = parse_ass_time(parts[2]) - start_time
            if end_ms <= 0: continue
            if start_ms < 0: start_ms = 0
            parts[1] = format_ass_time(start_ms)
            parts[2] = format_ass_time(end_ms)
            out.append(','.join(parts))
        else:
            out.append(line)

with open(f"{BASE_PATH}/legendas_temp.ass", "w", encoding="utf-8") as f:
    f.writelines(out)
print("Legendas ajustadas e cortadas para esta parte.")



with open(f"{BASE_PATH}/videorender-project.json", "r", encoding="utf-8") as f:
    config = json.load(f)

VIDEO_INPUT = f"{BASE_PATH}/pt2_enhanced.mp4"
AUDIO_INPUT = f"{BASE_PATH}/audio_dublado.mp3"
ASS_LOCAL = f"{BASE_PATH}/legendas_temp.ass"
OUTPUT_FILE = f"{BASE_PATH}/pt2_renderizado.mp4"

print("Arquivos prontos!")
print(f"  Video: {VIDEO_INPUT}")
print(f"  Audio: {AUDIO_INPUT}")
print(f"  ASS:   {ASS_LOCAL}")

cell_end(1, "done", "Download Arquivos do Drive concluido")


In [ ]:
cell_start(2, "Build Comando FFmpeg")

def build_ffmpeg_command(config, video_in, audio_in, ass_in, out_file, start_time=0):
    import os
    filters = []
    last_stream = "[0:v]"
    overlay_inputs = []
    
    out_format = config["video"].get("outputFormat", "16:9")
    out_w, out_h = (1080, 1920) if out_format == "9:16" else (1920, 1080)
    
    # 1. Base Crop / Zoom Logic
    sc = config.get("staticCrop", {})
    cz = config.get("cropZoom", {})
    
    if sc.get("enabled"):
        cx = sc.get("x", 0) / 100.0
        cy = sc.get("y", 0) / 100.0
        cw = sc.get("width", 100) / 100.0
        ch = sc.get("height", 100) / 100.0
        filters.append(f"{last_stream}crop=iw*{cw:.4f}:ih*{ch:.4f}:iw*{cx:.4f}:ih*{cy:.4f}[static_cropped]")
        last_stream = "[static_cropped]"
    elif cz.get("enabled"):
        zs = cz.get("zoomStart", 1.0)
        if zs >= 1.0:
            filters.append(f"{last_stream}scale={out_w}:{out_h}:force_original_aspect_ratio=increase,crop={out_w}:{out_h}[scaled]")
            last_stream = "[scaled]"
            if zs > 1.0:
                fx = cz.get("focusX", 0.5)
                fy = cz.get("focusY", 0.5)
                cw = int(out_w / zs)
                ch = int(out_h / zs)
                cx = int((out_w - cw) * fx)
                cy = int((out_h - ch) * fy)
                filters.append(f"{last_stream}crop={cw}:{ch}:{cx}:{cy},scale={out_w}:{out_h}[cropped]")
                last_stream = "[cropped]"
        else:
            filters.append(f"{last_stream}scale={out_w}:{out_h}:force_original_aspect_ratio=decrease[scaled]")
            bg_color = config.get("background", {}).get("solidColor", "#0a0a0a").replace("#", "")
            sm_w = int(out_w * zs)
            sm_h = int(out_h * zs)
            sm_w = sm_w - (sm_w % 2)
            sm_h = sm_h - (sm_h % 2)
            filters.append(f"[scaled]scale={sm_w}:{sm_h}[zoomedout]")
            filters.append(f"[zoomedout]pad={out_w}:{out_h}:(ow-iw)/2:(oh-ih)/2:color={bg_color}[padded]")
            last_stream = "[padded]"
            
    # 2. Position & Scaling
    vp = config.get("videoPosition", {})
    if vp.get("enabled"):
        scale_factor = vp.get("scale", 1.0)
        tx = vp.get("x", 0) / 100.0
        ty = vp.get("y", 0) / 100.0
        target_w = int(out_w * scale_factor)
        target_h = int(out_h * scale_factor)
        target_w = target_w - (target_w % 2)
        target_h = target_h - (target_h % 2)
        
        filters.append(f"{last_stream}scale={target_w}:{target_h}:force_original_aspect_ratio=decrease[scaled_vp]")
        bg_color = config.get("background", {}).get("solidColor", "#0a0a0a").replace("#", "")
        px = f"(ow-iw)/2+{int(tx*out_w)}"
        py = f"(oh-ih)/2+{int(ty*out_h)}"
        filters.append(f"[scaled_vp]pad={out_w}:{out_h}:{px}:{py}:color={bg_color}[positioned]")
        last_stream = "[positioned]"
    else:
        # Standard centering if staticCrop is enabled but no custom positioning
        if sc.get("enabled"):
            filters.append(f"{last_stream}scale={out_w}:{out_h}:force_original_aspect_ratio=decrease[scaled_sc]")
            bg_color = config.get("background", {}).get("solidColor", "#0a0a0a").replace("#", "")
            filters.append(f"[scaled_sc]pad={out_w}:{out_h}:(ow-iw)/2:(oh-ih)/2:color={bg_color}[padded_sc]")
            last_stream = "[padded_sc]"
        elif not cz.get("enabled"):
            # Default fallback when neither cropZoom nor staticCrop are active
            filters.append(f"{last_stream}scale={out_w}:{out_h}:force_original_aspect_ratio=decrease[scaled_std]")
            bg_color = config.get("background", {}).get("solidColor", "#0a0a0a").replace("#", "")
            filters.append(f"[scaled_std]pad={out_w}:{out_h}:(ow-iw)/2:(oh-ih)/2:color={bg_color}[padded_std]")
            last_stream = "[padded_std]"

    # 3. Geometric Transforms (Flip & Rotation) & Video Speed
    ve = config.get("videoEdit", {})
    if ve:
        if ve.get("hFlip"):
            filters.append(f"{last_stream}hflip[hflipped]")
            last_stream = "[hflipped]"
        if ve.get("vFlip"):
            filters.append(f"{last_stream}vflip[vflipped]")
            last_stream = "[vflipped]"
            
        rot = ve.get("rotate", 0)
        if rot == 90:
            filters.append(f"{last_stream}transpose=1[rotated]")
            last_stream = "[rotated]"
        elif rot == 180:
            filters.append(f"{last_stream}transpose=2,transpose=2[rotated]")
            last_stream = "[rotated]"
        elif rot == 270:
            filters.append(f"{last_stream}transpose=2[rotated]")
            last_stream = "[rotated]"
            
        speed = ve.get("speed", 1.0)
        if speed != 1.0:
            filters.append(f"{last_stream}setpts=PTS/{speed:.2f}[speed_v]")
            last_stream = "[speed_v]"

    # 4. Color Grading
    cg = config.get("colorGrade", {})
    if cg:
        b = cg.get("brightness", 0) / 100.0
        c = 1.0 + cg.get("contrast", 0) / 100.0
        s = 1.0 + cg.get("saturation", 0) / 100.0
        g = cg.get("gamma", 1.0)
        filters.append(f"{last_stream}eq=brightness={b}:contrast={c}:saturation={s}:gamma={g}[colorgraded]")
        last_stream = "[colorgraded]"
        temp = cg.get("temperature", 0)
        if temp != 0:
            red_mod = temp / 100.0 if temp > 0 else 0
            blue_mod = abs(temp) / 100.0 if temp < 0 else 0
            filters.append(f"{last_stream}colorbalance=rm={red_mod}:bm={blue_mod}[temp_applied]")
            last_stream = "[temp_applied]"
        sharp = cg.get("sharpness", 1.0)
        if sharp > 1.0:
            amount = sharp - 1.0
            filters.append(f"{last_stream}unsharp=5:5:{amount}:5:5:0.0[sharpened]")
            last_stream = "[sharpened]"
        v = cg.get("vignette", 0)
        if v > 0:
            filters.append(f"{last_stream}vignette=a={v}[vignetted]")
            last_stream = "[vignetted]"

    # 5. Overlays (Image/Text)
    overlays = config.get("overlays", [])
    for i, ov in enumerate(overlays):
        if ov["type"] in ["image", "watermark"]:
            header, encoded = ov["content"].split(",", 1)
            ext = header.split(";")[0].split("/")[1]
            filename = f"/kaggle/working/temp_overlay_{i}.{ext}"
            with open(filename, "wb") as f:
                f.write(base64.b64decode(encoded))
            overlay_inputs.append(filename)
            input_idx = len(overlay_inputs) + 1
            ox = int((ov["x"] / 100.0) * out_w)
            oy = int((ov["y"] / 100.0) * out_h)
            ow = int((ov["width"] / 100.0) * out_w)
            oh = int((ov["height"] / 100.0) * out_h)
            opacity = ov.get("opacity", 1.0)
            filters.append(f"[{input_idx}:v]scale={ow}:{oh}[ov_scaled_{i}]")
            alpha_filter = f",colorchannelmixer=aa={opacity}" if opacity < 1.0 else ""
            filters.append(f"[ov_scaled_{i}]format=rgba{alpha_filter}[ov_alpha_{i}]")
            time_filter = ""
            tin = ov.get("timeIn", 0) - start_time
            tout = ov.get("timeOut", 0) - start_time
            if ov.get("timeOut", 0) == 0: tout = 999999
            if tout <= 0: continue
            if tin < 0: tin = 0
            time_filter = f":enable='between(t,{tin},{tout})'"
            filters.append(f"{last_stream}[ov_alpha_{i}]overlay=x={ox}:y={oy}{time_filter}[with_ov_{i}]")
            last_stream = f"[with_ov_{i}]"
        elif ov["type"] == "text":
            ox = int((ov["x"] / 100.0) * out_w)
            oy = int((ov["y"] / 100.0) * out_h)
            txt = ov["content"].replace("'", "\\\'").replace(":", "\\\\:")
            fsize = int((ov.get("fontSize", 32) * (out_h / 1080)))
            fcolor = ov.get("fontColor", "#ffffff")
            
            ffamily = ov.get("fontFamily", "Montserrat")
            fweight = ov.get("fontWeight", "normal")
            fstyle = ov.get("fontStyle", "normal")
            
            suffix = "-Regular"
            if fweight in ["bold", "900"]: suffix = "-Bold"
            if fweight == "900" and ffamily == "Montserrat": suffix = "-Black"
            if fstyle == "italic": suffix = "-Italic"
            fontfile = f"/usr/share/fonts/truetype/custom/{ffamily}{suffix}.ttf"
            font_opt = f"fontfile='{fontfile}'"
            
            bg_color = ov.get("bgColor", "")
            bg_op = ov.get("bgOpacity", 0.5)
            box_filter = f":box=1:boxcolor={bg_color.replace('#','')}@{bg_op}:boxborderw={int(fsize*0.2)}" if bg_color else ""
            
            shadow_color = ov.get("shadowColor", "")
            shadow_filter = f":shadowcolor={shadow_color.replace('#','')}:shadowx={ov.get('shadowX', 2)}:shadowy={ov.get('shadowY', 2)}" if shadow_color else ""
            
            time_filter = ""
            tin = ov.get("timeIn", 0) - start_time
            tout = ov.get("timeOut", 0) - start_time
            if ov.get("timeOut", 0) == 0: tout = 999999
            if tout <= 0: continue
            if tin < 0: tin = 0
            time_filter = f":enable='between(t,{tin},{tout})'"
            
            filters.append(f"{last_stream}drawtext={font_opt}:text=\'{txt}\':x={ox}:y={oy}:fontsize={fsize}:fontcolor={fcolor}{box_filter}{shadow_filter}{time_filter}[with_txt_{i}]")
            last_stream = f"[with_txt_{i}]"

    # 6. Blur Band
    bb = config.get("blurBand", {})
    if bb.get("enabled"):
        bb_h = int((bb.get("height", 20) / 100) * out_h)
        bb_y = int((bb.get("positionY", 85) / 100) * out_h - bb_h / 2)
        bb_y = max(0, min(out_h - bb_h, bb_y))
        bb_blur = bb.get("blurIntensity", 20)
        bb_feather = bb.get("feather", 40)
        bb_color = bb.get("color", "#000000").replace("#", "")
        bb_opacity = bb.get("opacity", 0.6)
        
        filters.append(f"{last_stream}split[bb_main][bb_src]")
        filters.append(f"[bb_src]boxblur={bb_blur}:{bb_blur}[bb_blurred_only]")
        if bb_opacity > 0:
            filters.append(f"[bb_blurred_only]drawbox=x=0:y=0:w=iw:h=ih:color={bb_color}@{bb_opacity}:t=fill[bb_blurred]")
        else:
            filters.append(f"[bb_blurred_only]copy[bb_blurred]")
            
        half_feather = max(1, bb_feather // 2)
        filters.append(f"color=black@0:s={out_w}x{out_h},format=yuva420p,geq=lum=0:a=\'if(between(Y,{bb_y}-{half_feather},{bb_y}),255*(Y-{bb_y}+{half_feather})/{half_feather},if(between(Y,{bb_y},{bb_y}+{bb_h}),255,if(between(Y,{bb_y}+{bb_h},{bb_y}+{bb_h}+{half_feather}),255*({bb_y}+{bb_h}+{half_feather}-Y)/{half_feather},0)))\'[bb_mask]")
        filters.append(f"[bb_blurred][bb_mask]alphamerge[bb_masked]")
        filters.append(f"[bb_main][bb_masked]overlay=0:0[with_blur]")
        last_stream = "[with_blur]"

    filters.append(f"{last_stream}ass=\'legendas_temp.ass\'[subbed]")
    last_stream = "[subbed]"
    
    # 7. Audio Editing Filters (Volume, Speed, Fades)
    audio_filters = []
    last_audio = "[1:a]"
    if ve:
        speed = ve.get("speed", 1.0)
        if speed != 1.0:
            audio_filters.append(f"{last_audio}atempo={speed:.2f}[speed_a]")
            last_audio = "[speed_a]"
            
        vol = ve.get("volume", 100)
        if vol != 100:
            vol_factor = vol / 100.0
            audio_filters.append(f"{last_audio}volume={vol_factor:.2f}[vol_a]")
            last_audio = "[vol_a]"
            
        duration = config.get("video", {}).get("info", {}).get("duration", 0)
        if duration > 0:
            actual_duration = duration / speed
            fade_in = ve.get("audioFadeIn", 0)
            fade_out = ve.get("audioFadeOut", 0)
            if fade_in > 0:
                audio_filters.append(f"{last_audio}afade=t=in:ss=0:d={fade_in:.1f}[fadein_a]")
                last_audio = "[fadein_a]"
            if fade_out > 0 and actual_duration > fade_out:
                start_fade = actual_duration - fade_out
                audio_filters.append(f"{last_audio}afade=t=out:st={start_fade:.1f}:d={fade_out:.1f}[fadeout_a]")
                last_audio = "[fadeout_a]"

    if audio_filters:
        filters.extend(audio_filters)
        audio_stream = last_audio
    else:
        audio_stream = "1:a"
    
    filter_complex = ";".join(filters)
    cmd = ["ffmpeg", "-y", "-threads", "4", "-filter_threads", "4", "-i", video_in, "-ss", str(start_time), "-i", audio_in]
    for ov_file in overlay_inputs:
        cmd.extend(["-i", ov_file])
    cmd.extend(["-filter_complex", filter_complex, "-map", last_stream, "-map", audio_stream,
        "-c:v", "h264_nvenc", "-cq", "18", "-preset", "p6", "-c:a", "aac", "-b:a", "192k", "-shortest", out_file])
    return cmd

os.chdir(BASE_PATH)
command = build_ffmpeg_command(config, VIDEO_INPUT, AUDIO_INPUT, ASS_LOCAL, OUTPUT_FILE, start_time)
print("Comando FFmpeg gerado!")

cell_end(2, "done", "Build Comando FFmpeg concluido")

In [ ]:
cell_start(3, "Renderizacao")

print("Iniciando renderizacao...")
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True)
for line in process.stderr:
    if "frame=" in line:
        print(line.strip(), end="\r")
process.wait()
if process.returncode == 0:
    print("\n\nRenderizacao concluida!")
else:
    print("\n\nErro na renderizacao!")
    print(process.stderr.read() if hasattr(process.stderr, "read") else "")

cell_end(3, "done", "Renderizacao concluido")

In [ ]:
cell_start(4, "Upload Resultado")

salvar_no_drive(OUTPUT_FILE, f"{DRIVE_RENDER}/pt2_renderizado.mp4")
report_step("done", "Renderizacao PT2 concluida")
print("Upload concluido!")

cell_end(4, "done", "Upload Resultado concluido")